# Python程序设计（项目实战） · 工单P2.2《NumPy故障特征提取系统》 — 教师参考版
第4周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


工作目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/03-Python程序设计-项目实战


## 参考实现 `p22`（含依赖导入）

In [2]:
import sys
import numpy as np  # 通用科学计算


def p22():
    """P2.2 参考实现: 30维故障特征(每维含物理含义) → 200信号(100正常/100磨损)
    → 计时 → Z-score标准化 → 保存 features.npy + labels.npy。满足验收
    及格(D)=30特征正确+向量化为主, 优秀(B+)=<100ms+特征字典含物理含义。"""
    import time

    _SPC = {}                                      # FFT缓存: 同一信号只算一次

    def spectrum(x, fs=100):
        # 同一信号会被多个频域特征重复调用; 按 id(x) 缓存(30特征评估期内
        # 该数组对象存活, id稳定) → 每条信号只算一次FFT, 这是达标<100ms的关键
        key = (id(x), fs)
        cached = _SPC.get(key)
        if cached is not None:
            return cached
        amp = np.abs(np.fft.rfft(x)) * 2 / len(x)
        res = (np.fft.rfftfreq(len(x), 1 / fs), amp)
        if len(_SPC) > 8192:
            _SPC.clear()
        _SPC[key] = res
        return res

    def _sk(a):                                   # 偏度
        return ((a - a.mean()) ** 3).mean() / (a.std() ** 3 + 1e-12)

    def _ku(a):                                   # 峭度(减3)
        return ((a - a.mean()) ** 4).mean() / (a.std() ** 4 + 1e-12) - 3

    def _rolloff(fa, ratio=0.85):                 # 累积能量滚降频率
        f, a = fa
        c = np.cumsum(a ** 2) / ((a ** 2).sum() + 1e-12)
        return f[min(np.searchsorted(c, ratio), len(f) - 1)]

    # ============ 30维特征字典: 每维 = (函数, 物理含义) ============
    # 前21维复用教师版既有实现, 后9维按学生TODO清单补齐 → 合计30维
    FEATURES = {
        # ---- 时域幅值 ----
        "mean": (lambda x, fs: float(x.mean()), "均值: 振动能量平均水平"),
        "std": (lambda x, fs: float(x.std()), "标准差: 波动离散程度"),
        "max": (lambda x, fs: float(np.abs(x).max()), "峰值: 最大振幅, 对冲击敏感"),
        "ptp": (lambda x, fs: float(np.ptp(x)), "峰峰值: max-min, 磨损冲击幅值"),
        "rms": (lambda x, fs: float(np.sqrt((x ** 2).mean())), "均方根: 有效值, 能量代表"),
        "var": (lambda x, fs: float(x.var()), "方差: 信号强度离散度量"),
        "abs_mean": (lambda x, fs: float(np.abs(x).mean()), "绝对均值: 整流平均, 去符号偏移"),
        # ---- 时域无量纲 ----
        "crest": (lambda x, fs: float(np.abs(x).max() / np.sqrt((x ** 2).mean())),
                  "峰值指标: peak/rms, 早期冲击最敏感"),
        "kurtosis": (lambda x, fs: float(((x - x.mean()) ** 4).mean() / x.std() ** 4 - 3),
                     "峭度: 四阶标准矩, 冲击性指标"),
        "clearance": (lambda x, fs: float(np.abs(x).max() / (np.sqrt(np.abs(x)).mean() ** 2)),
                      "裕度指标: 对早期点蚀/剥落敏感"),
        "skewness": (lambda x, fs: float(((x - x.mean()) ** 3).mean() / x.std() ** 3),
                     "偏度: 分布不对称性, 剥离偏心"),
        "cv": (lambda x, fs: float(x.std() / abs(x.mean())), "变异系数: std/|mean| 相对波动"),
        "shape_factor": (lambda x, fs: float(np.sqrt((x ** 2).mean()) / np.abs(x).mean()),
                         "波形因子: rms/绝对均值, 波形饱满度"),
        "impulse_factor": (lambda x, fs: float(np.abs(x).max() / (np.abs(x).mean() + 1e-12)),
                           "脉冲因子: peak/绝对均值, 冲击强度"),
        "zero_crossing_rate": (lambda x, fs: float(np.mean(np.diff(np.sign(x)) != 0)),
                               "过零率: 符号变化占比, 高频/复杂度指标"),
        # ---- 频域 ----
        "dom_freq": (lambda x, fs: float(spectrum(x, fs)[0][spectrum(x, fs)[1].argmax()]),
                     "主频率: 最大幅值对应频率"),
        "f2": (lambda x, fs: float(sorted(spectrum(x, fs)[1])[-2]), "第2峰幅值: 次大谱峰高度"),
        "f3": (lambda x, fs: float(sorted(spectrum(x, fs)[1])[-3]), "第3峰幅值: 第三大谱峰高度"),
        "spec_energy": (lambda x, fs: float((spectrum(x, fs)[1] ** 2).sum()), "频谱总能量"),
        "spec_centroid": (lambda x, fs:
                          float((spectrum(x, fs)[0] * spectrum(x, fs)[1]).sum()
                                / (spectrum(x, fs)[1].sum() + 1e-12)),
                          "频域质心: 谱能量中心频率"),
        "spec_var": (lambda x, fs: float(spectrum(x, fs)[1].var()), "频域方差: 谱分布离散程度"),
        "bandwidth": (lambda x, fs:
                      float((lambda hi: hi.max() - hi.min())(
                          spectrum(x, fs)[0][spectrum(x, fs)[1] > spectrum(x, fs)[1].max() / 2])),
                      "带宽: 半高谱宽(幅值>半峰)"),
        "half_power_bw": (lambda x, fs:
                          float((lambda hi: hi.max() - hi.min())(
                              spectrum(x, fs)[0][spectrum(x, fs)[1] ** 2
                                                  > (spectrum(x, fs)[1] ** 2).max() / 2])),
                          "半功率带宽: 功率谱>半峰的频率跨度"),
        "energy_ratio_0_20hz": (lambda x, fs:
                                float((lambda f, a: (a[f < 20] ** 2).sum()
                                       / ((a ** 2).sum() + 1e-12))(*spectrum(x, fs))),
                                "低频能量占比: <20Hz谱能量 / 总能量"),
        "spec_rolloff": (lambda x, fs: float(_rolloff(spectrum(x, fs), 0.85)),
                         "频谱滚降点: 累积85%能量对应频率"),
        "rms_freq": (lambda x, fs:
                     float(np.sqrt((spectrum(x, fs)[0] ** 2 * spectrum(x, fs)[1]).sum()
                                   / (spectrum(x, fs)[1].sum() + 1e-12))),
                     "均方根频率: 能量在频率轴上的离散"),
        "spectral_flatness": (lambda x, fs:
                              float(np.exp(np.log(spectrum(x, fs)[1] + 1e-12).mean())
                                    / (spectrum(x, fs)[1].mean() + 1e-12)),
                              "频谱平坦度: 几何/算术均值, 谐波纯度"),
        "spectral_skewness": (lambda x, fs: float(_sk(spectrum(x, fs)[1])),
                              "频谱偏度: 幅值谱分布不对称性"),
        "spectral_kurtosis": (lambda x, fs: float(_ku(spectrum(x, fs)[1])),
                              "频谱峭度: 幅值谱冲击集中度"),
        # ---- 信息熵 ----
        "entropy": (lambda x, fs:
                    float(-np.nansum((p := np.abs(x) / np.abs(x).sum()) * np.log(p + 1e-12))),
                    "信息熵: 时域幅值分布复杂度"),
    }
    assert len(FEATURES) == 30, f"特征数应=30, 实际 {len(FEATURES)}"

    def make_signals(n=200, length=1000, fs=100):
        """合成两类信号: 正常(低幅+白噪) / 轴承磨损(冲击+高频), 与学生版语义一致"""
        rng = np.random.RandomState(42)
        t = np.arange(length) / fs
        sigs, labels = [], []
        for i in range(n):
            base = 0.5 + 0.1 * np.sin(2 * np.pi * 8 * t)
            s = base + rng.normal(0, 0.05, length)
            if i >= n // 2:                          # 磨损: 加冲击与高频
                s += 1.2 * np.abs(np.sin(2 * np.pi * 33 * t)) * rng.binomial(1, 0.08, length)
                s += 0.4 * np.sin(2 * np.pi * 60 * t)
            sigs.append(s)
            labels.append(0 if i < n // 2 else 1)
        return np.array(sigs), np.array(labels), fs

    names = list(FEATURES)
    sigs, labels, fs = make_signals()
    t0 = time.time()
    X = np.array([[FEATURES[n][0](s, fs) for n in names] for s in sigs])   # (200, 30)
    ms = (time.time() - t0) * 1000
    assert X.shape == (200, 30), X.shape
    X = (X - X.mean(0)) / (X.std(0) + 1e-9)                               # Z-score标准化
    np.save("features.npy", X)
    np.save("labels.npy", labels)

    grade = "优秀 ✓" if ms < 50 else ("及格 ✓" if ms < 100 else "需优化 ✗")
    print(f"特征矩阵 : {X.shape}  |  特征维度: {len(names)}  |  Z-score标准化完成")
    print(f"信号构成 : {len(sigs)} 条 (正常 {int((labels==0).sum())} / 磨损 {int((labels==1).sum())}), "
          f"{sigs.shape[1]} 点 @ {fs}Hz")
    print(f"计算耗时 : {ms:.1f}ms  (及格<100ms, 优秀<50ms)  → {grade}")
    print(f"已保存   : features.npy {X.shape} + labels.npy {labels.shape}  (cwd=03包根)")
    print("\n特征字典(物理含义) — 前8维示范(共30维, 满足'特征字典含物理含义'):")
    for n in names[:8]:
        print(f"    {n:20s} → {FEATURES[n][1]}")
    print("    ...(其余22维见上方 FEATURES 字典)")


## 学生版TODO → 教师答案对照
学生版starter中的TODO与上方参考实现（或下方双解）的对应关系：

| TODO | 题面（学生看到的指导） |
|------|----------------------|
| `?` | ---- 时域5个(TODO) ---- |
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） |
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） |
| `?` | ---- 频域15个(TODO, 先完成fft_help) ---- |
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） |
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） |
| `?` | ... TODO: 再补12个频域/统计特征, 例如: |
| `?` | TODO: Z-score标准化 → 保存 features.npy + labels.npy TODO: |
|  | 箱线图(matplotlib, 每类对比前6个特征) TODO: (超优)相关系数矩阵去冗余特征 |

> 发放纪律：本对照表**勿随学生版下发**；课堂按层放提示（D层演示照做/C层只给思路行）。

## 多解对比（已实跑互验）
| 解 | 路线 | 结论 |
|----|------|------|
| A | NumPy向量化30维 | P2.2主线（100ms内） |
| B | statistics/rolling自聚合 | 小样本/零依赖，一行得一大把特征 |

> 决策素材：特征要可讲→A；要快出基线→B。

## 执行与验证

In [3]:
p22()

# ---- 双解口径互验: NumPy向量化(解A, 本工单主线) vs statistics标准库(解B, 零依赖教学) ----
import importlib.util
_spec = importlib.util.spec_from_file_location("solutions_multi", "教师参考/solutions_multi.py")
_sm = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_sm)
_a, _b = _sm.W04_a(), _sm.W04_b()
print("\n--- 教师参考库 solutions_multi 双解口径互验 ---")
print("  解A NumPy向量化     [mean, std, ptp]:", _a)
print("  解B statistics标准库 [mean, std, ptp]:", _b)
print(f"  两解一致: mean={_a[0] == _b[0]}, ptp={_a[2] == _b[2]}, "
      "std 均为总体标准差(仅显示精度不同)")
print("  选型: 小样本/零依赖演示选B; 30维×万级量产主线选A(本工单向量化加法)")


特征矩阵 : (200, 30)  |  特征维度: 30  |  Z-score标准化完成
信号构成 : 200 条 (正常 100 / 磨损 100), 1000 点 @ 100Hz
计算耗时 : 89.2ms  (及格<100ms, 优秀<50ms)  → 及格 ✓
已保存   : features.npy (200, 30) + labels.npy (200,)  (cwd=03包根)

特征字典(物理含义) — 前8维示范(共30维, 满足'特征字典含物理含义'):
    mean                 → 均值: 振动能量平均水平
    std                  → 标准差: 波动离散程度
    max                  → 峰值: 最大振幅, 对冲击敏感
    ptp                  → 峰峰值: max-min, 磨损冲击幅值
    rms                  → 均方根: 有效值, 能量代表
    var                  → 方差: 信号强度离散度量
    abs_mean             → 绝对均值: 整流平均, 去符号偏移
    crest                → 峰值指标: peak/rms, 早期冲击最敏感
    ...(其余22维见上方 FEATURES 字典)

--- 教师参考库 solutions_multi 双解口径互验 ---
  解A NumPy向量化     [mean, std, ptp]: [3.0, 1.4142135623730951, 4.0]
  解B statistics标准库 [mean, std, ptp]: [3.0, 1.414214, 4.0]
  两解一致: mean=True, ptp=True, std 均为总体标准差(仅显示精度不同)
  选型: 小样本/零依赖演示选B; 30维×万级量产主线选A(本工单向量化加法)
